# Tworzymy nazwane klastry

W lekcji tej użyjemy LLM-ów do pomocy przy znalezieniu odpowiednich nazw i opisów dla klastrów.

In [15]:
import json
import pandas as pd
from openai import OpenAI
from dotenv import dotenv_values
from pycaret.clustering import predict_model, load_model

In [16]:
env = dotenv_values(".env")

openai_client = OpenAI(api_key=env["OPENAI_API_KEY"])

ładujemy nasze dane

In [19]:
df = pd.read_csv('welcome_survey_simple_v2.csv', sep=';')
df.fillna('Brak danych', inplace=True)
df.head()

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,Brak danych,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna


ładujemy model wytrenowany w poprzedniej lekcji

In [20]:
kmeans_pipeline = load_model('welcome_survey_clustering_pipeline_v2')

Transformation Pipeline and Model Successfully Loaded


aplikujemy model do danych

In [21]:
df_with_clusters = predict_model(model=kmeans_pipeline, data=df)
df_with_clusters["Cluster"].value_counts()

Cluster 0    42
Cluster 3    35
Cluster 2    33
Cluster 1    29
Cluster 5    26
Cluster 6    25
Cluster 7    24
Cluster 4    15
Name: Cluster, dtype: int64

stworzymy teraz prompt, który prześlemy do LLM-a w celu znalezienia odpowiednich nazw i opisów dla klastrów

In [22]:
cluster_descriptions = {}
for cluster_id in df_with_clusters['Cluster'].unique():
    cluster_df = df_with_clusters[df_with_clusters['Cluster'] == cluster_id]
    summary = ""
    for column in df_with_clusters:
        if column == 'Cluster':
            continue

        value_counts = cluster_df[column].value_counts()
        value_counts_str = ', '.join([f"{idx}: {cnt}" for idx, cnt in value_counts.items()])
        summary += f"{column} - {value_counts_str}\n"

    cluster_descriptions[cluster_id] = summary

In [23]:
print(cluster_descriptions["Cluster 7"])

age - 35-44: 18, 25-34: 4, 45-54: 2, 18-24: 0, 55-64: 0, <18: 0, >=65: 0, unknown: 0
edu_level - Wyższe: 24, Podstawowe: 0, Średnie: 0
fav_animals - Koty: 15, Brak ulubionych: 4, Inne: 3, Koty i Psy: 2, Psy: 0
fav_place - W górach: 19, Brak danych: 4, Inne: 1, Nad wodą: 0, W lesie: 0
gender - Mężczyzna: 20, Kobieta: 4, Brak danych: 0



In [24]:
prompt = "Użyliśmy algorytmu klastrowania."
for cluster_id, description in cluster_descriptions.items():
    prompt += f"\n\nKlaster {cluster_id}:\n{description}"

prompt += """
Wygeneruj najlepsze nazwy dla każdego z klasterów oraz ich opisy

Użyj formatu JSON. Przykładowo:
{
    "Cluster 0": {
        "name": "Klaster 0",
        "description": "W tym klastrze znajdują się osoby, które..."
    },
    "Cluster 1": {
        "name": "Klaster 1",
        "description": "W tym klastrze znajdują się osoby, które..."
    }
}
"""
print(prompt)

Użyliśmy algorytmu klastrowania.

Klaster Cluster 1:
age - 45-54: 19, 35-44: 6, unknown: 2, <18: 1, >=65: 1, 18-24: 0, 25-34: 0, 55-64: 0
edu_level - Wyższe: 26, Średnie: 2, Podstawowe: 1
fav_animals - Psy: 16, Brak ulubionych: 5, Inne: 4, Koty: 3, Koty i Psy: 1
fav_place - Brak danych: 9, Nad wodą: 8, W górach: 7, W lesie: 4, Inne: 1
gender - Kobieta: 28, Brak danych: 1, Mężczyzna: 0


Klaster Cluster 3:
age - 25-34: 10, 45-54: 9, 18-24: 8, 35-44: 7, >=65: 1, 55-64: 0, <18: 0, unknown: 0
edu_level - Średnie: 35, Podstawowe: 0, Wyższe: 0
fav_animals - Psy: 13, Koty: 11, Inne: 8, Brak ulubionych: 3, Koty i Psy: 0
fav_place - W górach: 18, Nad wodą: 6, W lesie: 6, Inne: 3, Brak danych: 2
gender - Mężczyzna: 33, Kobieta: 2, Brak danych: 0


Klaster Cluster 2:
age - 45-54: 22, 55-64: 7, 25-34: 2, 18-24: 1, >=65: 1, 35-44: 0, <18: 0, unknown: 0
edu_level - Wyższe: 33, Podstawowe: 0, Średnie: 0
fav_animals - Psy: 14, Inne: 11, Koty: 5, Brak ulubionych: 2, Koty i Psy: 1
fav_place - W lesie: 1

In [25]:
response = openai_client.chat.completions.create(
    model="gpt-4o",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }
    ],
)

In [26]:
result = response.choices[0].message.content.replace("```json", "").replace("```", "").strip()
cluster_names_and_descriptions = json.loads(result)

In [27]:
with open("welcome_survey_cluster_names_and_descriptions_v2.json", "w") as f:
    f.write(json.dumps(cluster_names_and_descriptions))

In [28]:
with open("welcome_survey_cluster_names_and_descriptions_v2.json", "r") as f:
    print(json.loads(f.read()))

{'Cluster 0': {'name': 'Miłośnicy Wody i Psów', 'description': 'W tym klastrze znajdują się głównie mężczyźni w wieku 35-44 lat z wyższym wykształceniem, którzy preferują spędzanie czasu nad wodą i mają psy jako ulubione zwierzęta.'}, 'Cluster 1': {'name': 'Kobiety z Wyższym Wykształceniem', 'description': 'W tym klastrze dominują kobiety w wieku 45-54 lat z wyższym wykształceniem, które preferują psy jako ulubione zwierzęta i mają różnorodne preferencje co do miejsc spędzania czasu.'}, 'Cluster 2': {'name': 'Mężczyźni z Wyższym Wykształceniem i Miłośnicy Lasów', 'description': 'W tym klastrze znajdują się mężczyźni w wieku 45-54 lat z wyższym wykształceniem, którzy preferują spędzanie czasu w lesie i mają różnorodne preferencje co do ulubionych zwierząt.'}, 'Cluster 3': {'name': 'Młodsi Mężczyźni ze Średnim Wykształceniem', 'description': 'W tym klastrze dominują młodsi mężczyźni w wieku 25-34 lat ze średnim wykształceniem, którzy preferują spędzanie czasu w górach i mają różnorodne p